# State Space. Blind Search.

Examples of *Reflex Agent* - Bird Flocking:

- [Bird Flocking Animation 1](../../videos/bird-flocking-1.mp4)
- [Bird Flocking Animation 2](../../videos/bird-flocking-2.mp4)


## 1. State Space

### 1.1 Definitions

In the first part of the course, we will take a look at very special type of agent - **Problem-Solving Agents**.

Problem-solving agents, decide what to do by finding sequence of actions that lead to desire state.

But what is a **problem**? A problem is defined by 4 things:

- *Initial state* - The state from which the agent begin. It is labeled as S.
- *Successor function* - This function has a type ```Fn(State) -> [(Action, State)]```. It accepts a state as an input and
returns list of pairs which first element is an action and the second is a child state.
- *Goal test* - This function has a type ```Fn(State) -> bool```. It checks if a state is a goal state. A goal state is labeled as G (G1, G2, G3 if there is more than one)
- *Path cost* - This function has a type ```Fn(Path) -> R```. It gives a numeric cost of path. R is the set of all real numbers.

🔑 The *initial state* and the *successor function* of a given problem implicitly define its **State Space**.

A *path* in a **state space** is an alternating sequence of a states and actions. It begins and ends with a state.

A *solution* to a problem is *path* in its **state space** which begins with the initial state and ends with a goal state.

A *solution* is called *optimal solution* if it has the smallest path cost among all solutions.

### 1.2 Representation of a state space

The *state space* can be represented as tree or graph which are called *Search Tree* or *Search Graph* respectively.
Each state is a node(vertex) and the action is an edge.

In *Search Tree*:

- The **initial state** is the root of the tree.
- The terminal and goal states are leaves.

### 1.3 Search Strategies Evaluation

We will use 4 main criteria to evaluate a strategy:

- **Completeness** - does it always find a solution if one exists?
- **Optimality** - does it always find a least-cost solution?
- **Time Complexity** - number of nodes generated/expanded
- **Space Complexity** - maximum number of nodes in memory


We will use 3 terms to measure time and space complexity:

- *b* - the maximum branching factor
- *d* - the depth of the optimal solution
- *m* - the maximum depth of the space

Example with a search tree:

![Search tree example with b = 3, d = 2, m = 3](../../images/bdm-tree-example.svg)



### 1.4 Global Search vs Local Search

- *Global Search* strategies - They look all over the state space. If necessary, all states will be traversed.
- *Local Search* strategies - They only look at the local area, so they can only look at states in this area.
If the solution is outside of it, they will not be able to find it.
    - They are heuristic method to solve computationally hard optimization problems.
    - They are used when we care only about a solution but not path to a solution.

## 2. Blind Search

### 2.1 Blind (Uninformed) Search vs Heuristic (Heuristic) Search

- *Blind Search* strategies - use only the information available in the problem definition.
- *Heuristic Search* strategies - use information on the **goal state** which helps in more efficient searching.
This information is obtained by a heuristic function that estimates how close a state is to the goal state.

### 2.2 Blind Search Examples

- *DFS* - **Depth-First Search**
- *BFS* - **Breadth-Frist Search**
- *UCS* - **Uniform-Cost Search**
- *DLS* - **Depth-Limited Search**
- *IDS* - **Iterative Deepening Search**

In [85]:
graph = {
    "A": ["B" , "S"],
    "B": ["A"],
    "S": ["A", "C", "G"],
    "C": ["D", "E", "F"],
    "G": ["S", "F", "H"],
    "D": ["C"],
    "E": ["C", "H"],
    "F": ["C", "G"],
    "H": ["E", "G"]
}

### 2.3 Depth-First Search

In [44]:
def dfs(state: str, visited: set[str], expanded: list[str]):
    if state in visited:
        return
    visited.add(state)
    expanded.append(state)
    for next_state in graph[state]:
        dfs(next_state, visited, expanded)

In [45]:
expanded = []
dfs("A", set(), expanded)
print("Expanded nodes - ", expanded)

Expanded nodes -  ['A', 'B', 'S', 'C', 'D', 'E', 'H', 'G', 'F']


| Complete | Optimal |       Time       |      Space        |
|:--------:|:-------:|:----------------:|:-----------------:|
|    No    |   No    | O(b<sup>m</sup>) |      O(b*m)       |

> *Complete* - fails in infinite-depth spaces, spaces with loops (but can be modified to avoid repeated states)

### 2.4 Breadth-First Search

In [46]:
from queue import Queue

def bfs(state: str, expanded: list[str]):
    visited = {state}
    q = Queue()
    q.put(state)
    while not q.empty():
        state = q.get()
        expanded.append(state)
        for next_state in graph[state]:
            if next_state not in visited:
                q.put(next_state)
                visited.add(next_state)


In [47]:
expanded = []
bfs("A", expanded)
print("Expanded nodes - ", expanded)

Expanded nodes -  ['A', 'B', 'S', 'C', 'G', 'D', 'E', 'F', 'H']


| Complete  |  Optimal  |        Time        |       Space        |
|:---------:|:---------:|:------------------:|:------------------:|
|   Yes*    |   Yes*    | O(b<sup>d+1</sup>) | O(b<sup>d+1</sup>) |

> *Complete* - Yes, if *b* is finite

> *Optimal* - if cost is 1 for every step

> *Time* - 1 + b + b<sup>2</sup> + b<sup>3</sup> + ... + b<sup>d</sup> + b*(b<sup>d</sup> - 1) = O(b<sup>d+1</sup>)

### 2.5 Uniform-Cost Search

In [62]:
weighted_graph = {
    "S": [("A", 7), ("B", 9), ("C", 14)],
    "A": [("S", 7), ("B", 10), ("D", 15)],
    "B": [("S", 9), ("A", 10), ("C", 2), ("D", 11)],
    "C": [("S", 14), ("B", 2), ("G", 9)],
    "D": [("A", 15), ("B", 11), ("G", 6)],
    "G": [("C", 9), ("D", 6)]
}

def is_goal_state_in_ucs(state: str) -> bool:
    return state == "G"

In [63]:
import heapq

class StateInfo:
    def __init__(self, state: str, path_cost: int | float, parent: StateInfo | None) -> None:
        self.state = state
        self.is_expanded = False
        self.path_cost = path_cost
        self.parent = parent


def ucs(state: str, expanded: list[str]) -> tuple[StateInfo | None, dict[str, StateInfo]]:
    states_info = {state: StateInfo(state, 0, None)}
    q = [(0, state)]
    while len(q) !=0 :
        path_cost, state = heapq.heappop(q)
        state_info = states_info[state]

        if state_info.is_expanded:
           continue
        state_info.is_expanded = True
        expanded.append(state)

        if is_goal_state_in_ucs(state):
            return state_info, states_info

        for next_state, action_weight in weighted_graph[state]:
            if next_state not in states_info:
                states_info[next_state] = StateInfo(next_state, state_info.path_cost + action_weight, state_info)
                heapq.heappush(q, (states_info[next_state].path_cost, next_state))
            elif states_info[next_state].path_cost > state_info.path_cost + action_weight:
                states_info[next_state] = StateInfo(next_state, state_info.path_cost + action_weight, state_info)
                heapq.heappush(q, (states_info[next_state].path_cost, next_state))

    return None, states_info


In [64]:
def extract_solution(goal_state_info: StateInfo | None, states_info: dict[str, StateInfo]) -> list[str]:
    if goal_state_info is None:
        return []

    solution = []
    state_info = goal_state_info
    while state_info.parent is not None:
        solution.append(state_info.state)
        state_info = states_info[state_info.parent.state]
    solution.append(state_info.state)
    solution.reverse()
    return solution

In [65]:
expanded = []

goal_state_info, states_info = ucs("S", expanded)
solution = extract_solution(goal_state_info, states_info)

print("Path cost - ", goal_state_info.path_cost)
print("Solution - ", solution)
print("Expanded nodes - ", expanded)

Path cost -  20
Solution -  ['S', 'B', 'C', 'G']
Expanded nodes -  ['S', 'A', 'B', 'C', 'D', 'G']


| Complete  | Optimal  |         Time          |         Sp ace        |
|:---------:|:--------:|:---------------------:|:---------------------:|
|   Yes*    |   Yes    | O(b<sup>[C*/ε]</sup>) | O(b<sup>[C*/ε]</sup>) |

> *ε* - the minimum step cost in **state space**

> *C\** - the *path cost* of the optimal solution

> *Complete* - Yes, if step cost ≥ ε


### 2.6 Depth-Limited Search

In [81]:
def is_goal_state_in_dls(state: str) -> bool:
    return state == "H"


def dls(state: str, limit: int, expanded: list[str]) -> list[str] | None:
    if limit == -1:
        return None

    expanded.append(state)
    if is_goal_state_in_dls(state):
        return [state]

    for next_state in graph[state]:
        solution  = dls(next_state, limit - 1, expanded)
        if solution is not None:
            solution.append(state)
            return solution
    return None

In [82]:
expanded = []
solution = dls("A", 2, expanded)
if solution is not None:
    solution.reverse()
print("Solution - ", solution)
print("Expanded nodes - ", expanded)

Solution -  None
Expanded nodes -  ['A', 'B', 'A', 'S', 'A', 'C', 'G']


| Complete  | Optimal  |       Time       |  Space  |
|:---------:|:--------:|:----------------:|:-------:|
|   Yes*    |    No    | O(b<sup>l</sup>) | O(b*l)  |

> *Complete* - Yes, but only if l ≥ d


### 2.7 Iterative Deepening Search

In [83]:
def ids(state: str, expanded: list[str]) -> list[str]:
    limit = 0
    while True:
        solution = dls(state, limit, expanded)
        if solution is not None:
            solution.reverse()
            return solution
        limit += 1

In [84]:
expanded = []
solution = ids("A", expanded)
print("Solution - ", solution)
print("Expanded nodes - ", expanded)

Solution -  ['A', 'S', 'G', 'H']
Expanded nodes -  ['A', 'A', 'B', 'S', 'A', 'B', 'A', 'S', 'A', 'C', 'G', 'A', 'B', 'A', 'B', 'S', 'S', 'A', 'B', 'S', 'C', 'D', 'E', 'F', 'G', 'S', 'F', 'H']


| Complete  | Optimal  |       Time       |  Space  |
|:---------:|:--------:|:----------------:|:-------:|
|    Yes    |   Yes*   | O(b<sup>d</sup>) | O(b*d)  |

> *Optimal* - Yes, if step cost = 1 (Can be modified to explore uniform-cost tree)

> *Time* -  (d + 1)*b<sup>0</sup> + d*b<sup>1</sup> + (d - 1)*b<sup>2</sup> + ... + b<sup>d</sup> = O(b<sup>d</sup>)


### 2.8 Summary

| Strategy | Complete  |  Optimal  |         Time          |         Space         | Data Structure |
|:--------:|:---------:|:---------:|:---------------------:|:---------------------:|:--------------:|
|   DFS    |    No     |    No     |   O(b<sup>m</sup>)    |        O(b*m)         |     Stack      |
|   BFS    |   Yes*    |   Yes*    |  O(b<sup>d+1</sup>)   |  O(b<sup>d+1</sup>)   |     Queue      |
|   UCS    |   Yes*    |    Yes    | O(b<sup>[C*/ε]</sup>) | O(b<sup>[C*/ε]</sup>) | Priority Queue |
|   DLS    |   Yes*    |    No     |   O(b<sup>l</sup>)    |        O(b*l)         |     Stack      |
|   IDS    |    Yes    |   Yes*    |   O(b<sup>d</sup>)    |        O(b*d)         |     Stack      |



## 3. Leaping Frogs

[Leaping Frogs Demo](../../demos/frogs/main.py)